In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv(
    "../data/processed/gps_clustered.csv"
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

print("Dataset shape:", df.shape)

Dataset shape: (24730077, 21)


In [3]:
print(df.columns.tolist())

['user_id', 'latitude', 'longitude', 'altitude', 'date', 'time', 'timestamp', 'hour', 'day', 'weekday', 'month', 'is_weekend', 'previous_latitude', 'previous_longitude', 'previous_timestamp', 'distance_km', 'time_difference_seconds', 'speed_kmh', 'location', 'time_spent_minutes', 'cluster']


In [4]:
route_data = df[
    df["cluster"] != -1
].copy()

print(
    "Route records:",
    len(route_data)
)

Route records: 24730077


In [5]:
route_data = route_data.sort_values(
    ["user_id", "timestamp"]
)

route_data = route_data.reset_index(
    drop=True
)

route_data.head()

,user_id,latitude,longitude,altitude,date,time,timestamp,hour,day,weekday,...,is_weekend,previous_latitude,previous_longitude,previous_timestamp,distance_km,time_difference_seconds,speed_kmh,location,time_spent_minutes,cluster
0,0,39.984702,116.318417,492.0,2008-10-23,02:53:04,2008-10-23 02:53:04,2,23,Thursday,...,False,NaN,NaN,NaN,0.000000,0.0,0.000000,39.9847_116.3184,0.000000,9
1,0,39.984683,116.318450,492.0,2008-10-23,02:53:10,2008-10-23 02:53:10,2,23,Thursday,...,False,39.984702,116.318417,2008-10-23 02:53:04,0.002812,6.0,1.687092,39.9847_116.3184,0.100000,9
2,0,39.984686,116.318417,492.0,2008-10-23,02:53:15,2008-10-23 02:53:15,2,23,Thursday,...,False,39.984683,116.318450,2008-10-23 02:53:10,0.002812,5.0,2.024341,39.9847_116.3184,0.083333,9
3,0,39.984688,116.318385,492.0,2008-10-23,02:53:20,2008-10-23 02:53:20,2,23,Thursday,...,False,39.984686,116.318417,2008-10-23 02:53:15,0.002726,5.0,1.962995,39.9847_116.3184,0.083333,9
4,0,39.984655,116.318263,492.0,2008-10-23,02:53:25,2008-10-23 02:53:25,2,23,Thursday,...,False,39.984688,116.318385,2008-10-23 02:53:20,0.010395,5.0,7.484055,39.9847_116.3183,0.083333,9


In [6]:
route_data["next_cluster"] = (
    route_data
    .groupby("user_id")["cluster"]
    .shift(-1)
)

In [7]:
route_data[
    [
        "user_id",
        "timestamp",
        "cluster",
        "next_cluster"
    ]
].head(20)

,user_id,timestamp,cluster,next_cluster
0,0,2008-10-23 02:53:04,9,9.0
1,0,2008-10-23 02:53:10,9,9.0
2,0,2008-10-23 02:53:15,9,9.0
3,0,2008-10-23 02:53:20,9,9.0
4,0,2008-10-23 02:53:25,9,9.0
5,0,2008-10-23 02:53:30,9,9.0
6,0,2008-10-23 02:53:35,9,9.0
7,0,2008-10-23 02:53:40,9,9.0
8,0,2008-10-23 02:53:45,9,9.0
9,0,2008-10-23 02:53:50,9,9.0


In [8]:
transitions = route_data[
    route_data["next_cluster"].notna()
].copy()

transitions = transitions[
    transitions["cluster"]
    != transitions["next_cluster"]
]

print(
    "Number of transitions:",
    len(transitions)
)

Number of transitions: 9412


In [9]:
transition_counts = (
    transitions
    .groupby(
        ["cluster", "next_cluster"]
    )
    .size()
    .reset_index(
        name="count"
    )
)

transition_counts.head(20)

,cluster,next_cluster,count
0,0,1.0,9
1,0,5.0,2
2,0,8.0,1
3,0,9.0,111
4,1,0.0,8
5,1,2.0,7
6,1,5.0,1
7,1,6.0,121
8,1,7.0,3
9,1,8.0,289


In [10]:
transition_counts[
    "probability"
] = (
    transition_counts["count"]
    /
    transition_counts
    .groupby("cluster")["count"]
    .transform("sum")
)

transition_counts.head(20)

,cluster,next_cluster,count,probability
0,0,1.0,9,0.073171
1,0,5.0,2,0.016260
2,0,8.0,1,0.008130
3,0,9.0,111,0.902439
4,1,0.0,8,0.001859
5,1,2.0,7,0.001626
6,1,5.0,1,0.000232
7,1,6.0,121,0.028113
8,1,7.0,3,0.000697
9,1,8.0,289,0.067147


In [11]:
transition_matrix = (
    transition_counts
    .pivot(
        index="cluster",
        columns="next_cluster",
        values="probability"
    )
    .fillna(0)
)

transition_matrix

next_cluster,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0
cluster,,,,,,,,,,
0,0.000000,0.073171,0.000000,0.000000,0.000000,0.016260,0.000000,0.000000,0.008130,0.902439
1,0.001859,0.000000,0.001626,0.000000,0.000000,0.000232,0.028113,0.000697,0.067147,0.900325
2,0.000000,0.048387,0.000000,0.000000,0.096774,0.000000,0.032258,0.016129,0.475806,0.330645
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000,0.800000
4,0.000000,0.000000,0.472222,0.000000,0.000000,0.000000,0.000000,0.000000,0.055556,0.472222
5,0.142857,0.142857,0.000000,0.071429,0.000000,0.000000,0.000000,0.000000,0.000000,0.642857
6,0.004587,0.440367,0.004587,0.000000,0.000000,0.004587,0.000000,0.009174,0.389908,0.146789
7,0.000000,0.062500,0.187500,0.000000,0.000000,0.000000,0.125000,0.000000,0.250000,0.375000
8,0.002242,0.643498,0.141256,0.002242,0.004484,0.000000,0.161435,0.006726,0.000000,0.038117


In [12]:
transition_matrix.to_csv(
    "../data/processed/transition_matrix.csv"
)

print(
    "Transition matrix saved!"
)

Transition matrix saved!


In [13]:
def predict_route(
    start_area,
    transition_matrix,
    steps=4
):
    
    route = [start_area]
    current_area = start_area
    
    for _ in range(steps):
        
        if current_area not in transition_matrix.index:
            break
        
        probabilities = (
            transition_matrix
            .loc[current_area]
        )
        
        probabilities = (
            probabilities[
                probabilities > 0
            ]
        )
        
        if len(probabilities) == 0:
            break
        
        next_area = (
            probabilities
            .idxmax()
        )
        
        route.append(
            next_area
        )
        
        current_area = next_area
    
    return route

In [14]:
start_area = int(
    route_data["cluster"].iloc[-1]
)

print(
    "Starting area:",
    start_area
)

Starting area: 0


In [15]:
predicted_route = predict_route(
    start_area,
    transition_matrix,
    steps=4
)

print(
    "Predicted route:",
    predicted_route
)

Predicted route: [0, np.float64(9.0), np.float64(1.0), np.float64(9.0), np.float64(1.0)]


In [16]:
route_text = " → ".join(
    [
        f"Area {int(x)}"
        for x in predicted_route
    ]
)

print(
    "Probable Route:"
)

print(route_text)

Probable Route:
Area 0 → Area 9 → Area 1 → Area 9 → Area 1


In [17]:
route_details = []

for i in range(
    len(predicted_route) - 1
):
    
    current = predicted_route[i]
    next_area = predicted_route[i + 1]
    
    probability = transition_matrix.loc[
        current,
        next_area
    ]
    
    route_details.append({
        "From_Area": current,
        "To_Area": next_area,
        "Probability": round(
            probability * 100,
            2
        )
    })

route_details = pd.DataFrame(
    route_details
)

route_details

,From_Area,To_Area,Probability
0,0.0,9.0,90.24
1,9.0,1.0,94.67
2,1.0,9.0,90.03
3,9.0,1.0,94.67


In [18]:
def next_area_probabilities(
    current_area,
    transition_matrix,
    top_n=5
):
    
    if current_area not in transition_matrix.index:
        return pd.DataFrame()
    
    probabilities = (
        transition_matrix
        .loc[current_area]
        .sort_values(
            ascending=False
        )
    )
    
    probabilities = (
        probabilities[
            probabilities > 0
        ]
        .head(top_n)
    )
    
    result = pd.DataFrame({
        "Next_Area": probabilities.index,
        "Probability": (
            probabilities.values * 100
        )
    })
    
    result["Probability"] = (
        result["Probability"]
        .round(2)
    )
    
    return result

In [19]:
next_areas = next_area_probabilities(
    start_area,
    transition_matrix,
    top_n=5
)

next_areas

,Next_Area,Probability
0,9.0,90.24
1,1.0,7.32
2,5.0,1.63
3,8.0,0.81


In [20]:
route_summary = pd.DataFrame({
    "Step": range(
        len(predicted_route)
    ),
    
    "Area": predicted_route
})

route_summary

,Step,Area
0,0,0.0
1,1,9.0
2,2,1.0
3,3,9.0
4,4,1.0


In [21]:
os.makedirs(
    "../data/processed",
    exist_ok=True
)

In [22]:
route_summary.to_csv(
    "../data/processed/predicted_route.csv",
    index=False
)

route_details.to_csv(
    "../data/processed/route_transition_probabilities.csv",
    index=False
)

print(
    "Route prediction results saved!"
)

Route prediction results saved!


In [23]:
import joblib

joblib.dump(
    transition_matrix,
    "../models/markov_route_model.pkl"
)

print(
    "Markov Chain model saved successfully!"
)

Markov Chain model saved successfully!


In [24]:
print(
    "========== ROUTE PREDICTION =========="
)

print(
    "Starting Area:",
    start_area
)

print(
    "\nProbable Route:"
)

print(
    route_text
)

print(
    "\nTransition Probabilities:"
)

display(
    route_details
)

========== ROUTE PREDICTION ==========
Starting Area: 0

Probable Route:
Area 0 → Area 9 → Area 1 → Area 9 → Area 1

Transition Probabilities:


,From_Area,To_Area,Probability
0,0.0,9.0,90.24
1,9.0,1.0,94.67
2,1.0,9.0,90.03
3,9.0,1.0,94.67
